# Classical music title tagger

Fine-tunes `bert-base-multilingual-cased` as a token classifier over the
`<CO> <CA> <K> <I> <N> <V> <D> <M> <S> <O>` title tags.

Training data is mounted from `data/train.txt` and `data/test.txt`; the trained
model is written to the `titles` volume at `/mnt/titles/fine_tuned_ner_model`.

In [ ]:
import modal

app = modal.App("classical-tagger")

image = (
    modal.Image.debian_slim(python_version="3.10")
    .pip_install("torch", "transformers", "datasets", "accelerate")
    .add_local_file("data/train.txt", "/root/data/train.txt")
    .add_local_file("data/test.txt", "/root/data/test.txt")
)

vol = modal.Volume.from_name("titles")

In [ ]:
import re

SPACE_TOKEN = "[unused1]"
model_name = "bert-base-multilingual-cased"
model_dir = "/mnt/titles/fine_tuned_ner_model"

tag_map = {
    "<CO>": 0, "<CA>": 1, "<K>": 2, "<I>": 3, "<N>": 4,
    "<V>": 5, "<D>": 6, "<M>": 7, "<S>": 8, "<O>": 9,
}
id2tag = {v: k for k, v in tag_map.items()}

In [ ]:
def parse_file(filepath):
    """Parses lines like <N>Name<S> <D>: into tokens and label_ids."""
    data = []
    pattern = re.compile(r"(<[A-Z]+>)([^<]+)?")

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            tokens, labels = [], []
            for tag, text in pattern.findall(line):
                if tag not in tag_map:
                    continue

                if tag == "<S>":
                    tokens.append(SPACE_TOKEN)
                    labels.append(tag_map[tag])
                    continue

                words = text.split()
                for i, word in enumerate(words):
                    tokens.append(word)
                    labels.append(tag_map[tag])
                    if i != len(words) - 1:
                        tokens.append(SPACE_TOKEN)
                        labels.append(tag_map[tag])

            if tokens:
                data.append({"tokens": tokens, "ner_tags": labels})
    return data

In [ ]:
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            label_ids.append(-100 if word_idx is None else label[word_idx])
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
def compute_metrics(p):
    import numpy as np

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    flattened_preds = [p for sublist in true_predictions for p in sublist]
    flattened_labels = [l for sublist in true_labels for l in sublist]
    accuracy = sum([1 if p == l else 0 for p, l in zip(flattened_preds, flattened_labels)]) / len(flattened_labels)

    return {"accuracy": accuracy}

In [ ]:
@app.function(image=image, volumes={"/mnt/titles": vol}, gpu="T4", timeout=3600)
def train():
    from datasets import Dataset
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        DataCollatorForTokenClassification,
        Trainer,
        TrainingArguments,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.add_special_tokens({"additional_special_tokens": [SPACE_TOKEN]})

    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(tag_map), id2label=id2tag, label2id=tag_map
    )

    ds_train = Dataset.from_list(parse_file("/root/data/train.txt"))
    ds_test = Dataset.from_list(parse_file("/root/data/test.txt"))

    tokenized_train = ds_train.map(lambda ex: tokenize_and_align_labels(ex, tokenizer), batched=True)
    tokenized_test = ds_test.map(lambda ex: tokenize_and_align_labels(ex, tokenizer), batched=True)

    args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=64,
        weight_decay=0.01,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    vol.commit()

    return trainer.evaluate()

In [ ]:
if __name__ == "__main__":
    with app.run():
        print(train.remote())